In [94]:
import boto3
from datetime import datetime, timezone

sm_client = boto3.client("sagemaker", region_name=region)

response = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName="Drift-Monitor"
)

print("Schedule Status :", response["MonitoringScheduleStatus"])
print("Schedule Expr   :", response["MonitoringScheduleConfig"]["ScheduleConfig"]["ScheduleExpression"])

last_run = response.get("LastMonitoringExecutionSummary", {})
if last_run:
    print("Last Run Status :", last_run.get("MonitoringExecutionStatus"))
    print("Last Run Time   :", last_run.get("ScheduledTime"))
else:
    print("Last Run        : No executions yet")

# Show current time so you can calculate next run
now = datetime.now(timezone.utc)
print("\n🕐 Current UTC Time:", now.strftime("%H:%M:%S"))
print("⏭️  Next run will be at the next top of the hour e.g.", 
      now.replace(minute=0, second=0, microsecond=0).strftime("%H:00:00 UTC"))

Schedule Status : Scheduled
Schedule Expr   : cron(0 * ? * * *)
Last Run Status : Failed
Last Run Time   : 2026-04-05 16:00:00+00:00

🕐 Current UTC Time: 17:06:12
⏭️  Next run will be at the next top of the hour e.g. 17:00:00 UTC


In [88]:
predictor = Predictor(
    endpoint_name="Driftplacement-model-endpointt",
    sagemaker_session=session,
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer()
)
print("✅ Connected to endpoint:", predictor.endpoint_name)

✅ Connected to endpoint: Driftplacement-model-endpointt


In [91]:
import pandas as pd

train_df = pd.read_csv(f"s3://{bucket}/train/train.csv")

# Normal traffic
print("📤 Sending normal traffic...")
for _, row in train_df.iterrows():
    predictor.predict(f"{row['cgpa']},{row['iq']},{row['profile_score']}")
print(f"✅ Sent {len(train_df)} normal requests")

# Abnormal traffic
print("📤 Sending abnormal traffic...")
for _ in range(50):
    predictor.predict("15.0,200.0,99.0")
print("✅ Abnormal traffic sent!")

📤 Sending normal traffic...
✅ Sent 240 normal requests
📤 Sending abnormal traffic...
✅ Abnormal traffic sent!
